# Multi-Language Audio Retrieval for CodeSwitched DRC Speech

## Introduction

This project addresses a practical gap in multilingual speech technology for the eastern Democratic Republic of Congo (DRC): the lack of usable French-Swahili-Lingala code-switched datasets for retrieval and speech applications. Relevant multilingual encoders and audio-text retrieval models exist, but there is no native, ready-to-use trilingual code-switched corpus for this setting. Our work therefore focused first on creating data, then on building a first retrieval benchmark over that data.

We completed three concrete pieces of work. First, we synthesized a trilingual code-switched corpus and released it on huggingface [synthesis_outputs](https://huggingface.co/datasets/nnoukastephen/CodeSwitch-SW-LIN-FRA-synthetic-audio). Second, we started a community collection web application in [lilics](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/tree/master/lilics) to support future real data acquisition. Third, we implemented a baseline experimentation pipeline in [experimentation.ipynb](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/blob/master/experimentation.ipynb), and exported the results to [experiment_outputs](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/tree/master/experiment_outputs).

The main question for this phase was deliberately modest: can we construct a reproducible synthetic code-switched dataset and verify that a simple retrieval baseline can operate on it in a controlled benchmark setting? The answer is yes, but with important limitations. The resulting conclusions apply to the synthetic corpus we generated, not to spontaneous DRC conversational speech in general.

## Methodology

### Datasets and Why They Were Used

We used a combination of source corpora, synthesized outputs, and auxiliary infrastructure.

The primary text dataset for experiments is the synthetic corpus in [synthesis_outputs/text/cs_text_dataset.jsonl](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/blob/master/synthesis_outputs/cs_text_dataset.jsonl). It contains 9,000 text records split into 7,200 train, 900 dev, and 900 test examples. Each record includes token-level language tags, matrix language, switch points, and quality metadata. This dataset is the main experimental corpus because it is the only resource we generated that explicitly models French-Swahili-Lingala code-switching.

We have pushed the corresponding audio to huggingface at `https://huggingface.co/datasets/nnoukastephen/CodeSwitch-SW-LIN-FRA-synthetic-audio`. The summary manifest reports 623 aligned clips split into 498 train, 62 dev, and 63 test items.

The source datasets were chosen because no naturally occurring trilingual code-switched corpus was available. The intended backbone sources were Gamayun Congolese Swahili-French, Google WaxalNLP, and Common Voice French. However, we found that the Hugging Face release of CLEAR-Global/Gamayun-kits does not expose aligned parallel pairs in the form we needed, so we manually downloaded the parallel corpus for [french-lingala](https://gamayun.translatorswb.org/download/gamayun-mini-kit-5k-lingala-french/) and [french-swahili](https://gamayun.translatorswb.org/download/gamayun-mini-kit-5k-swc-fra/) from [Gamayun community website](https://gamayun.translatorswb.org/data/).

We also prepared a real-data collection track. The application described and scafollded in [lilics](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/tree/master/lilics) is meant to collect future natural code-switched speech. The current frontend already supports consent, prompt presentation, and recording. This matters because it gives us a path beyond synthetic data.

### Text Synthesis Pipeline

First, seed examples are normalized and tokenized. Next, a rule-constrained generator applies Matrix Language Frame style substitutions, keeping Swahili as the dominant matrix language while inserting French and Lingala spans at controlled points. Then a copy-switch bootstrap stage adds lexical variation so the outputs are not trivial templates. Finally, filtered outputs are retained only if they satisfy quality constraints such as code-switch acceptability and consistency. The resulting corpus stores language tags and switch metadata at the token level.

An example from the released text corpus is:

> Leo tuko na probleme ya maji, mairie ilisema travaux zinaanza kesho matin.

This single sentence illustrates the intended behavior: the grammatical frame is mostly Swahili, but French nouns and discourse content are embedded, with Lingala also appearing elsewhere in the corpus. This is exactly the type of input the downstream retrieval benchmark is meant to test.

### Audio Synthesis and Audio Acquisition Strategy

The audio strategy combines two tracks. The first is synthetic generation from multilingual span rendering and quality filtering. The second is future collection of real data. For real data, we scaffolded the initial Lilics web app and defined a contributor workflow with consent, prompt selection, recording, metadata submission, and review. This supports the fact that synthetic data alone is insufficient for external validity.

We also investigated external audio sources. The project plan documents Mozilla Data Collective and Common Voice French as usable references, and WaxalNLP as the closest available Lingala and Swahili speech backbone. We did exploratory work around podcast-style Lingala audio sourcing, but no podcast corpus was incorporated into the released benchmark because licensing, segmentation, and provenance controls were not yet resolved. The current release therefore relies on synthetic audio plus a community collection path, rather than claiming a finalized podcast ingestion pipeline.

### Retrieval Benchmark Construction

The experimentation pipeline in [experimentation.ipynb](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/blob/master/experimentation.ipynb) implements a controlled text-to-text retrieval benchmark before attempting audio-text retrieval.

For the benchmark we used the split file in [synthesis_outputs/manifests/splits.json](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/blob/master/synthesis_outputs/manifests/splits.json). We derived a source-family identifier from the synthesis trace and used this lineage to define gold pairs while checking for leakage across train, dev, and test. For each eligible test query, we built a fixed 50-item candidate set consisting of one gold target, nine hard negatives, and forty random negatives. The exported benchmark file [experiment_outputs/csv/benchmark_queries.csv](https://github.com/Nnouka/CodeSwitch-SW-LIN-FRA-synthetic-audio/blob/master/experiment_outputs/csv/benchmark_queries.csv) shows this setup explicitly.

Although the text test split contains 900 items, not all of them become retrieval queries. Only items with a valid paired gold target from the same synthetic lineage are retained, giving 164 benchmark queries. This is a stricter and more controlled evaluation than simply retrieving over the entire test set.

### Metrics and Statistical Testing

The experimentation plan lists several possible retrieval metrics; we chose MRR, Recall@1, and nDCG@5 as the primary metrics, plus CSRS as a robustness measure. These choices are appropriate for a small controlled retrieval benchmark.

MRR is the most informative single ranking metric because it measures how early the first correct answer appears. Recall@1 measures immediate usability. nDCG@5 reflects the quality of the top-ranked portion of the list when the correct result is not always first. CSRS is defined as the ratio of high-switch MRR to low-switch MRR:

$$
\text{CSRS} = \frac{\text{MRR}_{\text{high-switch}}}{\text{MRR}_{\text{low-switch}}}
$$

We also implemented paired bootstrap resampling for MRR and nDCG differences and McNemar’s test for Recall@1. However, only one real baseline model was actually executed in this milestone, so the significance-testing framework is implemented but not yet used for a meaningful model-versus-model comparison. The only executed comparison is a self-comparison sanity check, which correctly yields zero deltas and $p = 1.0$.

## Results

### Corpus Release

The first concrete result is the release of a synthetic trilingual corpus `https://huggingface.co/datasets/nnoukastephen/CodeSwitch-SW-LIN-FRA-synthetic-audio`. 

### Baseline Text-to-Text Retrieval

We executed TF-IDF word-bigram baseline retrieval model. This model is intentionally simple. It is not meant to compete with SERENGETI or AfriBERTa; instead, it verifies that the benchmark logic, candidate pools, metrics, and exports all work end to end.

The baseline results are:

- MRR = 0.5803
- Recall@1 = 0.4085
- nDCG@5 = 0.6232
- CSRS = 1.4277

These results indicate that even a lexical baseline can often retrieve the correct sibling example early in the ranking. This is expected because the benchmark pairs are drawn from related synthetic lineages, so lexical overlap remains informative.

Performance by switch band is:

![Performance by switch band](experiment_outputs/plots/baseline_mrr_by_switch_band.png)

- High-switch: MRR 0.8333, Recall@1 0.6667, nDCG@5 0.8770
- Low-switch: MRR 0.5837, Recall@1 0.4156, nDCG@5 0.6238
- Medium-switch: MRR 0.3976, Recall@1 0.1429, nDCG@5 0.5025

The high-switch numbers appear best, which is counterintuitive. The correct interpretation is not that the model is inherently strongest on high-switch text. Rather, the high-switch subset is very small, and the specific synthetic examples in that subset are likely easier than average. The medium-switch band is more plausibly the hardest condition in this benchmark.

Performance by Lingala presence 
![Performance by Lingala presence](experiment_outputs/plots/baseline_mrr_by_lingala_presence.png) 
this shows a modest drop when Lingala is present:

- No Lingala: MRR 0.5912, Recall@1 0.4154, nDCG@5 0.6309
- Has Lingala: MRR 0.5385, Recall@1 0.3824, nDCG@5 0.5938

This suggests that Lingala-bearing examples may be somewhat harder for a lexical baseline, which is reasonable given lower lexical regularity and the smaller volume of Lingala-bearing material.


### Error Analysis Setup

A representative failure pattern is lexical-topic confusion: the baseline retrieves a sentence that shares high-overlap words such as “Bajeti ya afya imeongezeka...” but misses the intended paired target because several synthetic sentences are near-duplicates with only a few swapped tokens. This is a useful result because it shows that the benchmark is working as designed: the negatives are hard, not random nonsense.

## Discussion

The main contribution of this milestone is not a state-of-the-art retrieval result. It is the construction of a usable experiment. The project now has a released trilingual synthetic corpus, a reproducible benchmark notebook, a baseline retrieval result, and the start of a real-data collection platform. That is meaningful progress because we identified that the missing dataset was a key blocker.

Some insights follow from the current results.

First, the synthetic text corpus is sufficient to support controlled retrieval experiments. 

Second, the current benchmark is still heavily shaped by synthetic data. The best-performing examples are often those with strong lexical overlap between paired siblings. This is useful for debugging but means the current scores should not be treated as evidence of real-world conversational retrieval.

## Conclusion

This phase of the project achieved the essential prerequisite for multilingual code-switched retrieval research: we produced a released French-Swahili-Lingala synthetic corpus, and a reproducible baseline retrieval benchmark. The synthesized release contains 9,000 text records and 623 audio clips with frozen splits and metadata. The initial TF-IDF text-to-text baseline reaches an MRR of 0.5803, Recall@1 of 0.4085, and nDCG@5 of 0.6232 on the controlled benchmark.
